In [ ]:
import os
from datetime import datetime

import mlflow
import pandas as pd
from dotenv import load_dotenv
from mlflow.tracking import MlflowClient

load_dotenv()

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "")
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "")
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD", "")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME", "")

print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment Name: {MLFLOW_EXPERIMENT_NAME}")

In [ ]:
# Set up MLflow client
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

# Get experiment by name
try:
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    print(f"Experiment: {experiment}")
    if experiment is None:
        raise ValueError(f"Experiment '{MLFLOW_EXPERIMENT_NAME}' not found")

    experiment_id = experiment.experiment_id
    print(f"Found experiment: {experiment.name} (ID: {experiment_id})")
except Exception as e:
    print(f"Error getting experiment: {e}")
    experiment_id = None

In [ ]:
# Search for traces in the experiment
trace_data = []
if experiment_id:
    try:
        traces = client.search_traces(
            locations=[experiment_id], max_results=200, order_by=["timestamp_ms DESC"]
        )
        # Convert traces to a more readable format
        import json

        for trace in traces:
            request_obj = None
            if isinstance(trace.data.request, str):
                request_obj = json.loads(trace.data.request)

            response_obj = trace.data.response

            # If it's a string, parse it
            if isinstance(response_obj, str):
                response_obj = json.loads(response_obj)

            # Get the trace breakdown (spans)
            spans = trace.data.spans

            span_names = [span.name for span in trace.data.spans]

            if isinstance(response_obj, dict) and "response" in response_obj:
                response_text = response_obj["response"]
            elif isinstance(response_obj, dict):
                response_text = str(response_obj)
            else:
                response_text = str(response_obj)

            tags = trace.info.tags if hasattr(trace.info, "tags") else {}
            trace_metadata = (
                trace.info.trace_metadata if hasattr(trace.info, "trace_metadata") else {}
            )
            session_id = trace_metadata.get("mlflow.trace.session") or tags.get(
                "mlflow.trace.session"
            )

            trace_info = {
                "request": request_obj,
                "response": response_text,
                "trace_id": trace.info.request_id,
                "timestamp": datetime.fromtimestamp(trace.info.timestamp_ms / 1000),
                "status": trace.info.status,
                "execution_time_ms": trace.info.execution_time_ms,
                "tags": tags,
                "session_id": session_id,
            }
            trace_data.append(trace_info)

        print(f"Fetched {len(trace_data)} traces")

    except Exception as e:
        print(f"Error searching traces: {e}")
        traces = []
else:
    print("Cannot search traces without valid experiment_id")

In [ ]:
from IPython.display import Markdown
from IPython.display import display as idisplay

df = pd.DataFrame(trace_data)

# --- Session Overview ---
has_session = df["session_id"].notna()
grouped_df = df[has_session].groupby("session_id")
session_sizes = grouped_df.size()

single_trace_sessions = (session_sizes == 1).sum()
multi_trace_sessions = (session_sizes > 1).sum()
multi_trace_count = session_sizes[session_sizes > 1].sum()
ungrouped_count = (~has_session).sum()
total_sessions = single_trace_sessions + multi_trace_sessions

report = "# Trace Analysis Report\n\n"
report += f"**Dataset:** {len(df)} traces from `{MLFLOW_EXPERIMENT_NAME}` experiment\n\n"
report += "---\n\n## Session Overview\n\n"
report += "| Category | Count | Traces |\n|----------|-------|--------|\n"
report += f"| Single-trace sessions | {single_trace_sessions} | {single_trace_sessions} |\n"
report += f"| Multi-trace sessions | {multi_trace_sessions} | {multi_trace_count} |\n"
report += f"| Ungrouped (no session ID) | — | {ungrouped_count} |\n"
report += f"| **Total** | **{total_sessions} sessions** | **{len(df)}** |\n\n"

# --- Session Size Distribution ---
report += "### Session Size Distribution\n\n"
report += "| Traces per Session | Sessions |\n|--------------------|----------|\n"
for size, count in session_sizes.value_counts().sort_index().items():
    report += f"| {size} | {count} |\n"
report += "\n"

# --- Execution Time Profile ---
exec_secs = df["execution_time_ms"] / 1000.0
buckets = [
    ("10–30s", (exec_secs >= 10) & (exec_secs < 30)),
    ("30–60s", (exec_secs >= 30) & (exec_secs < 60)),
    ("60–120s", (exec_secs >= 60) & (exec_secs < 120)),
    ("120–180s", (exec_secs >= 120) & (exec_secs < 180)),
    (">180s", exec_secs >= 180),
]

report += "### Execution Time Profile\n\n"
report += "| Bucket | Count |\n|--------|-------|\n"
for label, mask in buckets:
    report += f"| {label} | {mask.sum()} |\n"
report += f"\n- **Mean:** {exec_secs.mean():.0f}s\n"
report += f"- **Median:** {exec_secs.median():.0f}s\n"
report += f"- **Min:** {exec_secs.min():.1f}s\n"
report += f"- **Max:** {exec_secs.max():.1f}s\n\n"


idisplay(Markdown(report))

In [ ]:
# --- Query Category Classification ---


def classify_query(request):
    """Classify a trace request into a query category based on keywords."""
    if not request:
        return "Other"

    text = str(request).lower()

    categories = [
        ("Icinga Alert Investigation", ["icinga", "alert", "check_command", "hostalive"]),
        ("Cost/Billing", ["cost", "billing", "spend", "chargeback", "budget", "$"]),
        ("Deployment Query", ["deploy", "sandbox", "provision", "catalog"]),
        ("Continue Investigation", ["keep investigating", "continue", "dig deeper", "go deeper"]),
        ("Config Lookup (agnosticv)", ["agnosticv", "agnosticd", "config", "catalog_item"]),
        ("AAP Job Investigation", ["aap", "ansible", "job", "tower", "awx"]),
        ("OpenShift Investigation", ["openshift", "ocp", "pod", "container", "cluster", "node"]),
        ("Security Investigation", ["security", "access key", "iam", "suspicious", "unauthorized"]),
        ("Report Generation", ["report", "summary", "generate report"]),
        ("Acknowledgment/Thank You", ["thank", "thanks", "ok", "got it", "okay", "thank you"]),
        ("User Frustration/Complaint", ["frustrat", "why you not", "not working", "wrong"]),
    ]

    for category, keywords in categories:
        if any(kw in text for kw in keywords):
            return category

    return "Other"


df["query_category"] = df["request"].apply(classify_query)

category_counts = df["query_category"].value_counts()

cat_report = "## Query Categories\n\n"
cat_report += "| Category | Traces |\n|----------|--------|\n"
for category, count in category_counts.items():
    cat_report += f"| {category} | {count} |\n"
cat_report += f"\n**Total:** {category_counts.sum()} traces\n"

idisplay(Markdown(cat_report))